---

# Decision Trees: Predicting Loan Defaults

In this notebook we will showcase the decision tree machine learning technique. Decision trees work by learning a series of simple classification rules and organizing them into a tree, so that when data outside the test set is introduced, it follows a particular set of decisions through a tree which eventually classifies it.

Decision trees tend to be very popular due to their ease of interpretation and their ability to classify more complex datasets. They are made of up two components: nodes and branches. Nodes can further be divided into three types:
- Root node: This is the topmost node which acts as the input node for feature vectors.
- Decision node: These are nodes where decisions and classifications are evaluated. Multiple decision nodes can be (and often are) visited in succession.
- Leaf node: This node represents the final classification of a peice of data.

We will begin illustrating this technique by attempting to predict loan defaults using the sklearn decision tree library. Load the dataset from Kaggle and use the path to access the file.

---

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)


---

Use the path that was printed to pass the dataset into a dataframe and reduce the size for faster calculations.

---

In [ ]:
# You can download a sample from Kaggle, then read locally
df = pd.read_csv("accepted_2007_to_2018Q4.csv", low_memory=False)

# Reduce size for demo purposes (optional)
df = df.sample(10000, random_state=42)

---

Take a look at the dataframe's structure:

---

In [ ]:
df

---

Let's restructure the dataframe to only include features that we care about.

---

In [ ]:
features = ['loan_amnt', 'term', 'int_rate', 'grade', 'home_ownership', 'annual_inc', 'purpose']
target = 'loan_status'

df = df[features + [target]]
df

---

We can also change the target labels to define the target variable as either 1 or 0. Let's go ahead and remove rows with empty values and encode catagorical features numerically.

---

In [ ]:
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]
df['target'] = np.where(df['loan_status'] == 'Charged Off', 1, 0)

df.dropna(inplace=True)

# Encode categorical features
for col in ['term', 'grade', 'home_ownership', 'purpose']:
    df[col] = pd.factorize(df[col])[0]

df

---

Split the data into training and testing data.

---

In [ ]:
from sklearn.model_selection import train_test_split

X = df[features]
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, test_size=0.2, random_state=42)


---

Time to train our model! We can control the maximum depth of our tree to prevent overfitting. We can also pass a random state so that you can replicate the results exactly.

---

In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)



---

Test the model and print key performance metrics. We can also nicely print a copy of our tree to see exactly where it's making its decisions!

---

In [ ]:
y_pred = tree.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=X.columns, class_names=["Fully Paid", "Charged Off"],
          filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree for Lending Club Loan Default")
plt.show()



---

We can now inspect our model to see how it did. It's acheiving about 78% accuracy, which, while not perfect, is far better than randomly guessing. Since the dataset has too many dimensions to easily print, it's hard to see the decision boundaries for ourselves on a plane, but the decision trees are nice in that we can still see the exact "logical" decision boundary!

---